# 02 Inspektera innan vi läser in

Innan vi lägger data i en tabell tittar vi på hur svaret från källan faktiskt ser ut. Det tar några minuter och sparar mycket letande senare. Vi får veta vilka fält som finns, vilken typ värdena har och om något avviker från dokumentationen.

Vi börjar med Webperfs lista över kommuners och regioners webbplatser. Den finns på adressen `https://api.webperf.se/public-sector/` och är dokumenterad på [api.webperf.se](https://api.webperf.se). Datan är öppen under licensen CC BY-SA 4.0, så vi anger Webperf som källa när vi använder den.

## Hämta svaret

Paketet `httpx` hämtar en adress på webben, ungefär som en webbläsare gör. Vi anger en `timeout` i sekunder så att koden inte väntar för evigt om servern inte svarar.

In [ ]:
import httpx

URL = "https://api.webperf.se/public-sector/"
svar = httpx.get(URL, timeout=30)

## Statuskod och headers

Först kontrollerar vi statuskoden. Koden 200 betyder att allt gick bra. Koder som börjar på 4 betyder att något var fel i vår fråga och koder som börjar på 5 att något gick fel hos servern.

In [ ]:
svar.status_code

Svaret har också headers, alltså uppgifter om svaret som inte är själva innehållet. Headern `content-type` berättar vilket format innehållet har.

In [ ]:
svar.headers["content-type"]

Webperf begränsar hur många frågor vi får ställa under en viss tid. Tre headers berättar om gränsen, hur många frågor vi har kvar och hur många sekunder det dröjer innan räkningen börjar om. Det är bra att känna till innan vi börjar hämta data i en loop.

In [ ]:
for namn in ["x-ratelimit-limit", "x-ratelimit-remaining", "retry-after"]:
    print(namn, svar.headers.get(namn))

## Råtexten

Innan vi tolkar något tittar vi på de första tecknen av svaret precis som det kom. Hakparenteserna med `:500` betyder att vi bara visar de första 500 tecknen.

In [ ]:
svar.text[:500]

Det ser ut som JSON, ett vanligt textformat för data. Att Alingsås skrivs `Alings\\u00e5s` är inget fel. JSON kan skriva tecken som å med en kod, och när vi tolkar svaret blir det rätt igen. Krullparenteser `{}` omger ett objekt med namngivna fält och hakparenteser `[]` omger en lista.

## Tolka JSON

Med `svar.json()` gör `httpx` om texten till Pythons egna typer. Ett JSON-objekt blir en ordbok, `dict`, och en JSON-lista blir en `list`.

In [ ]:
innehall = svar.json()

type(innehall)

Vi skriver ut varje nyckel på översta nivån tillsammans med vilken typ värdet har.

In [ ]:
for nyckel, varde in innehall.items():
    print(nyckel, type(varde).__name__)

Två saker är värda att notera. Fältet `result_count` är text, `str`, trots att det är ett antal. Dokumentationen säger samma sak, att de äldre endpoints levererar tal som textsträngar. Vi behöver alltså göra om det till ett heltal innan vi kan räkna med det.

In [ ]:
innehall["result_count"], type(innehall["result_count"]).__name__

Med `int()` gör vi om texten till ett heltal. Sedan jämför vi med hur många poster listan `data` faktiskt innehåller. Funktionen `len` räknar antalet. Svaret `True` betyder att uppgifterna stämmer överens.

In [ ]:
int(innehall["result_count"]) == len(innehall["data"])

Det andra är tidsstämpeln. Den saknar uppgift om tidszon, så vi vet inte säkert om den anger svensk tid eller UTC. Jämför gärna med klockan när du kör cellen.

In [ ]:
innehall["timestamp"]

## En post i taget

Listan `data` innehåller en post per webbplats. Python räknar från noll, så `[0]` är den första posten.

In [ ]:
poster = innehall["data"]

poster[0]

Vi ser fem fält. Fältet `sameAs` pekar på samma organisation i Wikidata, vilket vi använder i notebook 04. Fältet `uri` slutar med Webperfs eget id-nummer för webbplatsen.

## Har alla poster samma fält?

En vanlig överraskning är att någon post saknar ett fält eller har ett extra. Vi samlar fältnamnen för varje post i en mängd, `set`. En mängd behåller bara unika värden. Blir resultatet en enda uppsättning har alla poster samma fält.

In [ ]:
uppsattningar = {tuple(sorted(post.keys())) for post in poster}

uppsattningar

## Vilka kategorier finns?

`Counter` räknar hur många gånger varje värde förekommer.

In [ ]:
from collections import Counter

Counter(post["category"] for post in poster)

Sverige har 290 kommuner men listan har 289. Dokumentationen förklarar varför. Gotland är både kommun och region, och här finns Gotland bara med som Region Gotland. Det behöver vi komma ihåg när vi senare jämför med källor som räknar Gotland som kommun.

In [ ]:
[post for post in poster if "Gotland" in post["title"]]

## Leta efter avvikelser

Vi letar efter värden som inte följer mönstret. Alla kommuner heter inte något med kommun, en del kallar sig stad.

In [ ]:
[post["title"] for post in poster if post["category"] == "Municipality" and "kommun" not in post["title"]]

Adresserna till webbplatserna borde vara startsidor. Vi letar efter adresser med fler än två snedstreck, alltså adresser som pekar på en undersida.

In [ ]:
[post["url"] for post in poster if post["url"].count("/") > 2]

Till sist kontrollerar vi att alla länkar till Wikidata har samma form. Funktionen `all` svarar `True` bara om villkoret gäller för varje post.

In [ ]:
all(post["sameAs"].startswith("https://www.wikidata.org/wiki/Q") for post in poster)

## Spara rådata

Vi sparar svaret precis som det kom, innan vi ändrat något. Då kan vi gå tillbaka och kontrollera vad källan faktiskt sa, även om datan hos Webperf har uppdaterats sedan dess.

Mappen `data/radata` skapas om den inte redan finns. `svar.content` är svaret som byte, alltså exakt det servern skickade.

In [ ]:
from pathlib import Path

RADATA = Path("..") / "data" / "radata"
RADATA.mkdir(parents=True, exist_ok=True)

fil = RADATA / "webperf-public-sector.json"
fil.write_bytes(svar.content)
fil

## Det här vet vi nu

Så här såg det ut när vi körde notebooken i oktober 2026. Källan uppdateras, så siffrorna kan skilja sig åt när du kör den.

- Svaret är JSON med fyra fält på översta nivån och 310 webbplatser i `data`.
- `result_count` är text och behöver göras om till heltal.
- Tidsstämpeln saknar tidszon.
- Alla poster har samma fem fält.
- Gotland finns med som region, inte som kommun.
- Fyra kommuner kallar sig stad och tre adresser pekar på en undersida.

Listan innehåller inga betyg. I nästa notebook hämtar vi Webperfs nyare endpoints där betygen finns med, och läser in dem i pandas.